# NMEA 0183 — GPS and GNSS

GPS receivers, AIS transponders and marine instruments talk NMEA 0183: ASCII lines like `$GPGGA,...*47` with an XOR checksum.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.6.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.6.0-preview.1"

## Parse a sentence

In [ ]:
using IoTCom.Net.Protocols.Nmea;
var gga = (GgaMessage)NmeaParser.Parse("$GPGGA,123519,4807.038,N,01131.000,E,1,08,0.9,545.4,M,46.9,M,,*47")!;
Console.WriteLine($"{gga.Latitude:0.0000}, {gga.Longitude:0.0000} · {gga.Satellites} satellites · {gga.AltitudeMeters} m");

## A simulated receiver and a reader
`GnssState` merges GGA, RMC, GSA and GSV into one fix.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Transports;
var link = new InMemoryTransportListener();
var receiver = NmeaServer.Create(o => o.ListenInMemory(link));
await receiver.StartAsync();
var reader = NmeaReader.Create(o => o.UseInMemory(link));
await reader.ConnectAsync();
await Task.Delay(100);

var sim = new NmeaSimulator();
foreach (var s in sim.GenerateEpoch(DateTimeOffset.UtcNow, TimeSpan.FromSeconds(30))) await receiver.BroadcastAsync(s);
await Task.Delay(200);
var fix = reader.Gnss.Current;
Console.WriteLine($"{fix.Latitude:0.00000}, {fix.Longitude:0.00000} · {fix.SpeedKmh} km/h · {fix.SatellitesInView.Count} in view");

## Troubleshooting
- No data on serial: most receivers default to 9600 or 4800 baud, 8N1.
- Fix but no position: check RMC status `A` (active) vs `V` (void).

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*